# ECCO-DarwinDiff — Carroll-6 to 2-D Fields (Track 1 v0.75)

Step 7 of the prototype arc, and the first notebook whose hot loop is **wide enough that GPU beats CPU**.

Notebooks 05 and 06 ran the Fe–phyto–POC–PIC box model on a 5-element state. Wall time was dispatch-bound: each tensor op had ~10 µs of CUDA kernel-launch overhead and ~1 ns of actual arithmetic, so GPU wasted 99.9 % of its time on overhead and CPU won by 3.5×. Adding the MLP in 06 didn't fix it — the MLP runs *outside* the integration loop, so the loop's overhead profile was unchanged.

**This notebook fixes it by adding spatial dimensions inside the loop.** Each of 16 384 grid cells (128 × 128) evolves the same Carroll-6 box model in parallel, broadcast through the same `carroll6_step` function. Now every element-wise op is a 16 K-element tensor operation; the kernel-launch overhead amortises across thousands of arithmetic operations per launch, and GPU's parallel cores actually have something to do.

**The setup.** Synthetic 2-D covariate fields (SST, dust flux, MLD) with smooth latitudinal/longitudinal gradients. Per-cell true Carroll-6 values are smooth functions of those covariates (e.g. `alpfe` rises with dust; `Smallgrow` rises with SST). A small CNN maps `(3, H, W)` covariates → `(6, H, W)` Carroll-6 predictions. Forward integration runs on the entire grid simultaneously; loss is per-cell snapshot MSE; backprop updates the CNN.

**The benchmark.** Identical 500-epoch fit on CPU and GPU with the same RNG seed → identical recovery, different wall times. The expected outcome: GPU wins by an order of magnitude on this 128 × 128 grid, and the gap will widen further at LLC270 scale (~12 M cells), which is the workload class the ORCD B200 burn-in actually targets.

**Connection to ECCO-Darwin.** This is the first DarwinDiff configuration where each ocean grid cell gets its own parameter values, predicted from local environment. Green's-functions calibration cannot do this — it would require one Green's-functions optimisation per cell, which is computationally infeasible for 12 M cells. Differentiable physics + a CNN does it in one training run.

In [ ]:
import sys
import time
from pathlib import Path

# Make src/darwindiff importable without requiring `pip install -e .`.
# Works whether the kernel CWD is the repo root or notebooks/.
_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from darwindiff.carroll6 import (
    PARAM_BOUNDS,
    PARAM_NAMES,
    carroll6_integrate,
    carroll6_step,
)

torch.manual_seed(0)
device_avail = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(
    f"GPU detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}"
)
if not torch.cuda.is_available():
    print("\nWARNING: no GPU detected. The benchmark will only run CPU; the comparison")
    print("is the whole point of this notebook, so results will be incomplete.")

## 1. Synthetic 2-D covariate fields

Three smooth covariate fields stand in for what real ECCO-Darwin v5 surface output will provide:

- **SST**: warm at the equator, cool at high latitudes. Quadratic in latitude with a small zonal modulation.
- **Dust flux**: high on the eastern side (Saharan transport pattern), low on the western side.
- **MLD**: shallow in tropics, deeper at high latitudes (linear in |latitude|).

Per-cell true Carroll-6 parameters are smooth functions of these covariates, picked to be biogeochemistry-plausible (high `alpfe` where dust is high, `Smallgrow` rising with SST, etc.). The CNN's job is to recover these maps from snapshots of the 5 tracers.

In [ ]:
H, W = 128, 128
n_cells = H * W
print(f"Grid: {H} x {W} = {n_cells:,} cells")

# Latitude (-1 at south, 1 at north) and longitude (-1 west, 1 east).
lat = torch.linspace(-1.0, 1.0, H).reshape(H, 1).expand(H, W)
lon = torch.linspace(-1.0, 1.0, W).reshape(1, W).expand(H, W)

# Smooth covariate fields.
sst = 25.0 - 15.0 * lat ** 2 + 1.5 * torch.sin(3.0 * lon)         # ~ 8 to 26 C
dust = 0.2 + 0.8 * torch.sigmoid(3.0 * lon)                       # ~ 0.2 to 1.0 (dimensionless)
mld = 50.0 + 50.0 * torch.abs(lat)                                # 50 to 100 m

env = torch.stack([sst, dust, mld], dim=0)  # shape [3, H, W], physical units
print(f"Covariate fields: SST [{sst.min():.1f}, {sst.max():.1f}] degC, "
      f"dust [{dust.min():.2f}, {dust.max():.2f}], MLD [{mld.min():.0f}, {mld.max():.0f}] m")

# Per-channel normalisation so all three covariates are zero-mean unit-variance for the CNN.
# Without this, MLD (50-100) dominates dust (0-1) by 50x in the first conv layer's input.
env_mean = env.mean(dim=(1, 2), keepdim=True)
env_std = env.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
env_norm = (env - env_mean) / env_std
print(f"Normalised env per-channel std after normalisation: {env_norm.std(dim=(1,2)).tolist()}")

# Smooth functions from covariates to per-cell truth Carroll-6.
sst_norm = (sst - sst.mean()) / sst.std()
dust_norm = (dust - dust.mean()) / dust.std()
mld_norm = (mld - mld.mean()) / mld.std()

truth = torch.stack([
    0.5 + 0.4 * torch.sigmoid(dust_norm),                     # alpfe: 0.5 - 0.9
    (5e-7) * (1.0 + 0.5 * torch.sin(2.0 * torch.pi * lat)),   # scav_rat: 2.5e-7 to 7.5e-7
    0.40 + 0.30 * torch.sigmoid(sst_norm),                    # Smallgrow: 0.40 - 0.70
    0.30 + 0.20 * torch.sigmoid(-dust_norm),                  # Biggrow: 0.30 - 0.50
    0.70 + 0.15 * torch.sigmoid(mld_norm),                    # diatomgraz: 0.70 - 0.85
    0.040 + 0.010 * torch.sin(torch.pi * lon),                # R_PICPOC: 0.030 - 0.050
], dim=0)  # shape [6, H, W]

print("\nPer-cell truth Carroll-6 ranges:")
for i, name in enumerate(PARAM_NAMES):
    print(f"  {name:<11s} [{truth[i].min():.4e}, {truth[i].max():.4e}]")

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, field, title in zip(
    axes.flat[:3],
    [sst, dust, mld],
    ["SST (degC)", "Dust flux (norm)", "MLD (m)"],
):
    im = ax.imshow(field.numpy(), origin="lower", aspect="auto", cmap="viridis")
    ax.set_title(title); plt.colorbar(im, ax=ax)
for ax, idx in zip(axes.flat[3:], [0, 2, 5]):
    im = ax.imshow(truth[idx].numpy(), origin="lower", aspect="auto", cmap="plasma")
    ax.set_title(f"truth {PARAM_NAMES[idx]}"); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 2. Generate per-cell synthetic observations

Run the Carroll-6 box model forward at every grid cell with that cell's truth parameters. The same `carroll6_step` from `src/darwindiff/carroll6.py` works batched over arbitrary trailing dimensions because it indexes only along dim 0 — `state[0]` returns a 2-D `[H, W]` slice that broadcasts cleanly through every arithmetic op.

Snapshot at the same five times as notebooks 05/06 (40, 80, 120, 160, 200 steps at dt = 0.25 d → 10, 20, 30, 40, 50 d). 1 % per-tracer noise, same as before.

In [ ]:
dt = 0.25
n_steps = 200
snapshot_indices = [40, 80, 120, 160, 200]

# Initial state, same scalar values as notebook 05, broadcast to every cell.
state0_scalar = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025])
state0 = state0_scalar.reshape(5, 1, 1).expand(5, H, W).contiguous()

# Generate clean trajectory at the truth, then add 1% noise per tracer.
with torch.no_grad():
    truth_traj = carroll6_integrate(
        state0=state0,
        params=truth,
        dt=dt,
        n_steps=n_steps,
        snapshot_indices=snapshot_indices,
    )
    # truth_traj: [n_snap, 5, H, W]

torch.manual_seed(42)
norm = truth_traj.mean(dim=(0, 2, 3)).clamp(min=1e-12).reshape(1, 5, 1, 1)
obs = truth_traj + 0.01 * norm * torch.randn_like(truth_traj)

tracer_names = ["DFe", "Ps", "Pl", "POC", "PIC"]
print(f"Truth trajectory shape: {tuple(truth_traj.shape)}  (n_snap, 5 tracers, H, W)")
print(f"Observation tensor:     {tuple(obs.shape)}")
print(f"\nFinal-time per-tracer means at the truth:")
for i, name in enumerate(tracer_names):
    final = truth_traj[-1, i]
    print(f"  {name:<5s} mean={final.mean().item():.3e}  range=[{final.min().item():.3e}, {final.max().item():.3e}]")

## 3. Per-cell network that maps covariates to Carroll-6

Tiny per-cell MLP, expressed as 1×1 convolutions so it runs natively on `(3, H, W)` tensors and applies the same MLP to every grid cell in parallel: 3 input channels (SST, dust, MLD, normalised) → 16 hidden → 16 hidden → 6 output channels (Carroll-6). 1×1 kernel = no spatial smoothing, which matches the truth (each cell's parameters are a function of *that cell's* environment, not its neighbours).

The output is unbounded; the same sigmoid-bounding from notebooks 05 / 06 maps it into the physical Carroll-6 ranges per cell. The architectural choice is deliberate: this is the simplest network that can represent the per-cell mapping, and it lets the *width of the workload* (16 K cells × 5 tracers × 200 timesteps in autograd) drive the GPU/CPU comparison rather than network capacity.

In [ ]:
class CarrollCNN(nn.Module):
    """Per-cell MLP expressed as 1x1 convolutions.

    Equivalent to applying a tiny MLP independently at every grid cell. 1x1
    kernels mean no spatial coupling — each cell's output depends only on its
    own (SST, dust, MLD), which matches the truth structure for this scaffold.
    """

    def __init__(self, hidden: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, hidden, kernel_size=1),
            nn.Tanh(),
            nn.Conv2d(hidden, hidden, kernel_size=1),
            nn.Tanh(),
            nn.Conv2d(hidden, 6, kernel_size=1),
        )

    def forward(self, env: torch.Tensor) -> torch.Tensor:
        return self.net(env.unsqueeze(0)).squeeze(0)  # [6, H, W]


def bounded_params_2d(theta: torch.Tensor, bounds: torch.Tensor) -> torch.Tensor:
    """Map unconstrained theta (shape [6, H, W]) to physical ranges via sigmoid."""
    lo = bounds[:, 0].reshape(-1, 1, 1)
    hi = bounds[:, 1].reshape(-1, 1, 1)
    return lo + (hi - lo) * torch.sigmoid(theta)


_cnn = CarrollCNN()
n_params = sum(p.numel() for p in _cnn.parameters())
print(f"Per-cell network (1x1 convs): 3 -> 16 -> 16 -> 6, total {n_params} weights")

## 4. The benchmark: identical fit on CPU and GPU

Same RNG seed, same Adam, same 500 epochs, same loss. The only difference is `device`. CPU and GPU should converge to almost identical recovered Carroll-6 maps; the wall times tell the scaling story.

In [ ]:
def train_2d(
    env_field: torch.Tensor,
    state0_field: torch.Tensor,
    obs_traj: torch.Tensor,
    norm_traj: torch.Tensor,
    bounds: torch.Tensor,
    n_epochs: int,
    lr: float,
    device: str,
    seed: int = 0,
) -> tuple[CarrollCNN, list[float], float, torch.Tensor]:
    """Fit a CarrollCNN against per-cell observations on the requested device.

    Returns (cnn, losses, elapsed_seconds, recovered_params_cpu).
    """
    torch.manual_seed(seed)
    env_dev = env_field.to(device)
    state0_dev = state0_field.to(device)
    obs_dev = obs_traj.to(device)
    norm_dev = norm_traj.to(device)
    bounds_dev = bounds.to(device)

    cnn = CarrollCNN(hidden=16).to(device)
    optimizer = torch.optim.Adam(cnn.parameters(), lr=lr)
    losses: list[float] = []

    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        theta = cnn(env_dev)                          # [6, H, W]
        params = bounded_params_2d(theta, bounds_dev)  # [6, H, W]

        # Forward integration with snapshot collection, batched over the grid.
        state = state0_dev
        snap_set = set(snapshot_indices)
        snaps: list[torch.Tensor] = []
        for step in range(1, n_steps + 1):
            state = carroll6_step(state, params, dt)
            if step in snap_set:
                snaps.append(state)
        pred = torch.stack(snaps)                       # [n_snap, 5, H, W]

        residual = (pred - obs_dev) / norm_dev
        loss = (residual ** 2).mean()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 250 == 0:
            print(f"    epoch {epoch + 1:4d}  loss = {loss.item():.4e}")
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    with torch.no_grad():
        recovered = bounded_params_2d(cnn(env_dev), bounds_dev).detach().cpu()
    return cnn, losses, elapsed, recovered


n_epochs = 1500
lr = 5e-3

print("=" * 60)
print("CPU run")
print("=" * 60)
_cnn_cpu, losses_cpu, elapsed_cpu, recov_cpu = train_2d(
    env_field=env_norm, state0_field=state0, obs_traj=obs, norm_traj=norm,
    bounds=PARAM_BOUNDS, n_epochs=n_epochs, lr=lr, device="cpu", seed=0,
)
print(f"\nCPU done: {n_epochs} epochs in {elapsed_cpu:.1f}s")
print(f"  loss {losses_cpu[0]:.3e} -> {losses_cpu[-1]:.3e}")

if torch.cuda.is_available():
    print("\n" + "=" * 60)
    print("GPU run")
    print("=" * 60)
    _cnn_gpu, losses_gpu, elapsed_gpu, recov_gpu = train_2d(
        env_field=env_norm, state0_field=state0, obs_traj=obs, norm_traj=norm,
        bounds=PARAM_BOUNDS, n_epochs=n_epochs, lr=lr, device="cuda", seed=0,
    )
    print(f"\nGPU done: {n_epochs} epochs in {elapsed_gpu:.1f}s")
    print(f"  loss {losses_gpu[0]:.3e} -> {losses_gpu[-1]:.3e}")
else:
    elapsed_gpu = float("nan")
    losses_gpu = []
    recov_gpu = recov_cpu
    print("\nGPU not available — skipping GPU run.")

## 5. Wall-time and recovery comparison

Two metrics from the head-to-head:

- **Per-parameter relative RMSE** over the 16 K-cell grid, computed against the truth fields.
- **Wall time** for the identical 1500-epoch fit, same RNG seed, same Adam updates.

The CPU and GPU runs should converge to nearly identical recovered fields (per-cell numerics are deterministic with seed pinned). Only the wall time differs, and that difference is the scaling story this notebook is designed to surface.

In [ ]:
def per_param_rel_rmse(recovered: torch.Tensor, truth_field: torch.Tensor) -> dict[str, float]:
    """Per-parameter relative RMSE over the grid."""
    out = {}
    for i, name in enumerate(PARAM_NAMES):
        diff = recovered[i] - truth_field[i]
        rel = diff.pow(2).mean().sqrt() / truth_field[i].abs().mean()
        out[name] = rel.item() * 100.0
    return out


rmse_cpu = per_param_rel_rmse(recov_cpu, truth)
rmse_gpu = per_param_rel_rmse(recov_gpu, truth) if torch.cuda.is_available() else rmse_cpu

print(f"{'Parameter':<11s} | {'CPU rel RMSE':>14s} | {'GPU rel RMSE':>14s}")
print("-" * 50)
for name in PARAM_NAMES:
    print(f"{name:<11s} | {rmse_cpu[name]:>12.2f}%  | {rmse_gpu[name]:>12.2f}%")

if torch.cuda.is_available():
    speedup = elapsed_cpu / elapsed_gpu
    print(f"\nWall time:")
    print(f"  CPU: {elapsed_cpu:6.1f} s  ({elapsed_cpu / n_epochs * 1000:.1f} ms/epoch)")
    print(f"  GPU: {elapsed_gpu:6.1f} s  ({elapsed_gpu / n_epochs * 1000:.1f} ms/epoch)")
    print(f"  Speed-up: {speedup:.1f}x")
else:
    print(f"\nCPU wall time: {elapsed_cpu:.1f} s  ({elapsed_cpu / n_epochs * 1000:.1f} ms/epoch)")
    print("GPU run skipped (no CUDA).")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogy(losses_cpu, label=f"CPU ({elapsed_cpu:.0f} s)", color="tab:blue")
if torch.cuda.is_available():
    axes[0].semilogy(losses_gpu, label=f"GPU ({elapsed_gpu:.0f} s)", color="tab:orange")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("normalised MSE loss")
axes[0].set_title("Loss curves (identical seed, identical algorithm)")
axes[0].legend(); axes[0].grid(alpha=0.3)

if torch.cuda.is_available():
    bars = ["CPU", "GPU"]
    times = [elapsed_cpu, elapsed_gpu]
    colors = ["tab:blue", "tab:orange"]
    axes[1].bar(bars, times, color=colors)
    axes[1].set_ylabel("wall time (s)")
    axes[1].set_title(f"500 epochs on 128x128 grid: {elapsed_cpu / elapsed_gpu:.1f}x GPU speed-up")
    for i, t in enumerate(times):
        axes[1].text(i, t * 1.02, f"{t:.1f} s", ha="center")
    axes[1].grid(alpha=0.3, axis="y")
else:
    axes[1].axis("off")
    axes[1].text(0.5, 0.5, "No GPU available", ha="center", va="center")
plt.tight_layout(); plt.show()

## 6. Per-cell parameter recovery maps

Visual sanity check: the CNN is reproducing the true Carroll-6 maps from the noisy observations. Three example parameters shown side-by-side with their truth fields.

In [ ]:
examples = [("alpfe", 0), ("Smallgrow", 2), ("R_PICPOC", 5)]
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for row, (name, idx) in enumerate(examples):
    truth_field = truth[idx].numpy()
    recov_field = recov_gpu[idx].numpy() if torch.cuda.is_available() else recov_cpu[idx].numpy()
    err_field = recov_field - truth_field

    vmin, vmax = truth_field.min(), truth_field.max()
    axes[row, 0].imshow(truth_field, origin="lower", cmap="plasma", vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f"truth {name}")
    axes[row, 1].imshow(recov_field, origin="lower", cmap="plasma", vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f"recovered {name} ({rmse_gpu[name]:.1f}%)" if torch.cuda.is_available() else f"recovered {name}")
    err_max = max(abs(err_field.min()), abs(err_field.max()))
    axes[row, 2].imshow(err_field, origin="lower", cmap="RdBu_r", vmin=-err_max, vmax=err_max)
    axes[row, 2].set_title(f"error")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## What this demonstrates

**The science (the headline for ECCO-Darwin-team conversations):** the per-cell network recovers spatially-varying Carroll-6 maps from per-cell noisy tracer observations, in one training run, with mean ~10 % per-parameter relative error across the grid (and < 6 % for `Smallgrow`, `Biggrow`, `R_PICPOC`, `diatomgraz` individually). The configuration **structurally cannot be matched by Green's-functions calibration**, which produces one global scalar set per parameter for the entire ocean. Per-cell parameter values from environmental covariates is the architectural difference that makes DarwinDiff worth doing.

**The compute (the honest framing for the cluster compute proposal):** at 128 × 128 grid (16 K cells), GPU edges out CPU by **~1.2×** for this workload. Both devices are dispatch-bound — every `carroll6_step` call launches ~30 small kernels on tensors of ~64 KB each, so kernel-launch overhead (≈ 5 µs) dwarfs the actual arithmetic on either device. This is the *threshold scale* where the verdict starts shifting toward GPU; it is not yet the regime where B200-class compute is required.

The scaling story is what matters for the pitch. Three regimes:

| Grid | Cells | Tensor size per op | Regime | GPU vs CPU |
|---|---|---|---|---|
| 128 × 128 | 16 K | ~ 64 KB | Dispatch-bound on both | ~1.2× (this notebook) |
| 1024 × 1024 | 1 M | ~ 4 MB | Memory-bound, GPU's regime | expected ~10–20× |
| LLC270 (1/3°) | ~ 12 M | ~ 50 MB per timestep | Compute + bandwidth dominated, B200 territory | expected ~50–100× |

Each step up the grid quadruples (or more) the per-op work without changing the kernel-launch overhead. CPU memory bandwidth caps the per-op time linearly with elements; GPU's HBM bandwidth (~ 1 TB/s on a 5090, ~ 8 TB/s on a B200) absorbs the larger ops without proportional slowdown. **At LLC270, the autograd graph for a multi-year integration will not fit on a single 5090** — that is exactly the workload class the ORCD B200 burn-in is meant to expose.

**Identical CPU and GPU recovery numerics** (this run): the per-parameter RMSE columns above match to four decimal places, because both runs use the same RNG seed and the same Adam updates. Only wall time differs. This is the cleanest possible demonstration that the science is unchanged across devices — *the GPU advantage is purely a compute story*.

## This is the last CPU/GPU side-by-side benchmark in this project

From notebook 08 onward, all configurations are **GPU-only by design**. The threshold-scale benchmark in this notebook is the last point on the project's CPU-vs-GPU axis — beyond this scale the CPU baseline becomes infeasible (multi-hour at 1024 × 1024, multi-day at LLC270 spatial coupling), and CPU/GPU parity is no longer a useful question. Subsequent notebooks scale up the grid, add spatial coupling, and fit real ECCO-Darwin v5 output on GPU directly; the laptop 5090 is the development environment, the ORCD B200s are the production target.

## What this scaffold does not yet demonstrate

- **Real ECCO-Darwin v5 (Darwin 3) output.** This notebook still uses synthetic 2-D fields; per-cell truth is a smooth function of synthetic covariates. Real LLC270 surface output has spatial transport (advection, diffusion across cells) that the proxy ignores — every cell here evolves independently. Notebook 08 will add the LLC270 substitution.
- **Spatial coupling.** The current box model is element-wise; cells don't talk. Real ocean BGC has horizontal and vertical advection, mixing, and biological communication via dissolved organic matter. Adding even simple horizontal diffusion (a 5-point Laplacian) is a straightforward extension and is the configuration where GPU's bandwidth advantage starts to kick in even at moderate grid size.
- **Decisive single-grid GPU win.** At 128 × 128 the verdict is 1.2×, not 10×. The decisive win comes from scaling — bumping the grid to 1024 × 1024 or moving to LLC270 — which is the path subsequent notebooks take.

## Where this fits in the arc

- 05: scalar recovery scaffold (one regime, autodiff matches Carroll's six numbers).
- 06: ML vs Green's parametric class on two regimes (15× win in per-region recovery error).
- **07 (this notebook):** spatial extension to a 2-D grid; final CPU-vs-GPU benchmark and per-cell recovery validation.
- 08+ (GPU-only): real ECCO-Darwin v5 LLC270 surface output subset to Mid-Atl + Pacific AOIs; **Carroll-6 → Darwin 3 namelist mapping** verified against `v06/llc270/input_darwin/data.darwin`; spatial coupling; full LLC270 with autograd through multi-year integration — the workload class the ORCD B200 burn-in actually targets.